# DamXer paper reproduction

This notebook is a thin, auditable wrapper around `scripts/reproduce.py`. The default path validates the released CSV bytes and recomputes the manuscript table from the published seed ledger. It does **not** relabel a smoke test or ledger check as fresh training.

In [ ]:
from pathlib import Path
import json, subprocess, sys

cwd = Path.cwd().resolve()
REPO = cwd if (cwd / 'scripts' / 'reproduce.py').is_file() else cwd.parent
assert (REPO / 'scripts' / 'reproduce.py').is_file(), 'Open this notebook from the DamXer repository.'
print('Repository:', REPO)
print('Python:', sys.version.split()[0])

## 1. One-click integrity and paper-table check

This checks all six public data files against the frozen manifest, then independently recomputes all nine five-seed aggregates.

In [ ]:
subprocess.run([sys.executable, str(REPO / 'scripts' / 'reproduce.py'), '--mode', 'check'], cwd=REPO, check=True)
report_path = REPO / 'artifacts' / 'reproduction' / 'check_report.json'
report = json.loads(report_path.read_text())
print('Status:', report['status'])
for name, stats in report['aggregates'].items():
    print(f"{name:35s} {stats['test']['mean']:.6f} ± {stats['test']['std']:.6f}")

## 2. Optional fresh training

Set `RUN_TRAINING=True` to let the repository wrapper create a local `.venv`, install the forecasting dependencies, and launch training. `damxer` runs 15 CUDA jobs; `paper` runs all 45 paper jobs and may take hours. Fresh GPU results are accepted within the declared aggregate tolerance, not by bitwise equality across hardware. Because executing a notebook changes its own Git file, notebook-launched runs explicitly record an allowed dirty worktree. Use `./reproduce.sh paper --device cuda` from a clean checkout for paper-grade provenance.

In [ ]:
RUN_TRAINING = False
MODE = 'damxer'   # use 'paper' for all nine settings
DEVICE = 'cuda'

if RUN_TRAINING:
    subprocess.run([str(REPO / 'reproduce.sh'), MODE, '--device', DEVICE, '--allow-dirty-code'], cwd=REPO, check=True)
else:
    print('Fresh training skipped. Set RUN_TRAINING=True to start it.')